In [26]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

In [28]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ahmed\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [29]:
df = pd.read_csv("customer_complaints_1.csv")
df.head()

,author,posted_on,rating,text
0,"Alantae of Chesterfeild, MI","Nov. 22, 2016",1,I used to love Comcast. Until all these consta...
1,"Vera of Philadelphia, PA","Nov. 19, 2016",1,I'm so over Comcast! The worst internet provid...
2,"Sarah of Rancho Cordova, CA","Nov. 17, 2016",1,If I could give them a negative star or no sta...
3,"Dennis of Manchester, NH","Nov. 16, 2016",1,I've had the worst experiences so far since in...
4,"Ryan of Bellevue, WA","Nov. 14, 2016",1,Check your contract when you sign up for Comca...


In [30]:
df.columns.tolist()

['author', 'posted_on', 'rating', 'text']

In [31]:
text_col = 'text'
texts = df[text_col].dropna().astype(str).tolist()

print("Using column:", text_col)
print("Number of documents:", len(texts))
print(texts[0])

Using column: text
Number of documents: 19
I used to love Comcast. Until all these constant updates. My internet and cable crash a lot at night, and sometimes during the day, some channels don't even work and on demand sometimes don't play either. I wish they will do something about it. Because just a few mins ago, the internet have crashed for about 20 mins for no reason. I'm tired of it and thinking about switching to Wow or something. Please do not get Xfinity.


In [32]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

In [33]:
clean_texts = [preprocess(text) for text in texts]

print("Original:")
print(texts[0])
print("\nPreprocessed:")
print(clean_texts[0])

Original:
I used to love Comcast. Until all these constant updates. My internet and cable crash a lot at night, and sometimes during the day, some channels don't even work and on demand sometimes don't play either. I wish they will do something about it. Because just a few mins ago, the internet have crashed for about 20 mins for no reason. I'm tired of it and thinking about switching to Wow or something. Please do not get Xfinity.

Preprocessed:
used love comcast constant update internet cable crash lot night sometimes day channel dont even work demand sometimes dont play either wish something min ago internet crashed min reason im tired thinking switching wow something please get xfinity


In [34]:
vectorizer = TfidfVectorizer()
X_tfidf = vectorizer.fit_transform(clean_texts)

k = 3
kmeans_tfidf = KMeans(n_clusters=k, random_state=42, n_init=10)
tfidf_labels = kmeans_tfidf.fit_predict(X_tfidf)

print("TF-IDF clustering completed.")

TF-IDF clustering completed.


In [35]:
order_centroids = kmeans_tfidf.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for i in range(k):
    print(f"Cluster {i}:")
    top_terms = [terms[ind] for ind in order_centroids[i, :10]]
    print(", ".join(top_terms))
    print()

Cluster 0:
comcast, service, time, would, customer, month, since, second, adding, im

Cluster 1:
contract, speed, internet, mbps, xfinity, customer, told, service, would, tech

Cluster 2:
rude, rep, service, day, joke, charge, local, fee, people, bill



In [36]:
df_tfidf = df.dropna(subset=[text_col]).copy()
df_tfidf['Cluster_TFIDF'] = tfidf_labels
df_tfidf[[text_col, 'Cluster_TFIDF']].head(10)

,text,Cluster_TFIDF
0,I used to love Comcast. Until all these consta...,1
1,I'm so over Comcast! The worst internet provid...,0
2,If I could give them a negative star or no sta...,0
3,I've had the worst experiences so far since in...,0
4,Check your contract when you sign up for Comca...,1
5,Thank God. I am changing to Dish. They gave me...,0
6,I Have been a long time customer and only have...,1
7,There is a malfunction on the DVR manager whic...,0
8,Charges overwhelming. Comcast service rep was ...,2
9,"I have had cable, DISH, and U-verse, etc. in t...",0


In [37]:
from gensim.models import Word2Vec

In [38]:
tokenized_texts = [text.split() for text in clean_texts]

In [39]:
word2vec_model = Word2Vec(
    sentences=tokenized_texts,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec model trained.")

Word2Vec model trained.


In [40]:
X_w2v = []
for doc in tokenized_texts:
    vectors = [word2vec_model.wv[word] for word in doc if word in word2vec_model.wv]
    if len(vectors) > 0:
        doc_vector = np.mean(vectors, axis=0)
    else:
        doc_vector = np.zeros(100)
    X_w2v.append(doc_vector)

X_w2v = np.array(X_w2v)
print("Word2Vec vectors shape:", X_w2v.shape)

Word2Vec vectors shape: (19, 100)


In [41]:
kmeans_w2v = KMeans(n_clusters=k, random_state=42, n_init=10)
w2v_labels = kmeans_w2v.fit_predict(X_w2v)

print("Word2Vec clustering completed.")

Word2Vec clustering completed.


C:\Users\ahmed\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [42]:
df_w2v = df.dropna(subset=[text_col]).copy()
df_w2v['Cluster_Word2Vec'] = w2v_labels
df_w2v[[text_col, 'Cluster_Word2Vec']].head(10)

,text,Cluster_Word2Vec
0,I used to love Comcast. Until all these consta...,1
1,I'm so over Comcast! The worst internet provid...,0
2,If I could give them a negative star or no sta...,0
3,I've had the worst experiences so far since in...,0
4,Check your contract when you sign up for Comca...,1
5,Thank God. I am changing to Dish. They gave me...,0
6,I Have been a long time customer and only have...,0
7,There is a malfunction on the DVR manager whic...,0
8,Charges overwhelming. Comcast service rep was ...,2
9,"I have had cable, DISH, and U-verse, etc. in t...",0


In [43]:
print("TF-IDF cluster counts:")
print(df_tfidf['Cluster_TFIDF'].value_counts())

print("\nWord2Vec cluster counts:")
print(df_w2v['Cluster_Word2Vec'].value_counts())

TF-IDF cluster counts:
Cluster_TFIDF
1    9
0    7
2    3
Name: count, dtype: int64

Word2Vec cluster counts:
Cluster_Word2Vec
0    16
1     2
2     1
Name: count, dtype: int64


In [44]:
df_result = df.dropna(subset=[text_col]).copy()
df_result['Cleaned_Text'] = clean_texts
df_result['Cluster_TFIDF'] = tfidf_labels
df_result['Cluster_Word2Vec'] = w2v_labels

df_result.head(10)

,author,posted_on,rating,text,Cleaned_Text,Cluster_TFIDF,Cluster_Word2Vec
0,"Alantae of Chesterfeild, MI","Nov. 22, 2016",1,I used to love Comcast. Until all these consta...,used love comcast constant update internet cab...,1,1
1,"Vera of Philadelphia, PA","Nov. 19, 2016",1,I'm so over Comcast! The worst internet provid...,im comcast worst internet provider im taking o...,0,0
2,"Sarah of Rancho Cordova, CA","Nov. 17, 2016",1,If I could give them a negative star or no sta...,could give negative star star review would nev...,0,0
3,"Dennis of Manchester, NH","Nov. 16, 2016",1,I've had the worst experiences so far since in...,ive worst experience far since install nothing...,0,0
4,"Ryan of Bellevue, WA","Nov. 14, 2016",1,Check your contract when you sign up for Comca...,check contract sign comcast advertised offer m...,1,1
5,"Terri of Mobile, AL","Nov. 9, 2016",1,Thank God. I am changing to Dish. They gave me...,thank god changing dish gave awesome pricing s...,0,0
6,"Kellie of Salt Lake City, UT","Nov. 9, 2016",1,I Have been a long time customer and only have...,long time customer xfinity isp local walmart n...,1,0
7,"Kathleen of New Haven, CT","Nov. 6, 2016",2,There is a malfunction on the DVR manager whic...,malfunction dvr manager preventing u adding re...,0,0
8,"Shira of Bloomfield, NJ","Nov. 5, 2016",1,Charges overwhelming. Comcast service rep was ...,charge overwhelming comcast service rep ignora...,2,2
9,"Kristy of Alpharetta, GA","Nov. 2, 2016",1,"I have had cable, DISH, and U-verse, etc. in t...",cable dish uverse etc past eh know comcast tak...,0,0


In [45]:
df_result.to_csv("customer_complaints_clustered.csv", index=False)
print("File saved as customer_complaints_clustered.csv")

File saved as customer_complaints_clustered.csv


In [46]:
for i in range(k):
    print(f"\n===== TF-IDF Cluster {i} Sample Documents =====")
    sample_docs = df_result[df_result['Cluster_TFIDF'] == i][text_col].head(5).tolist()
    for j, doc in enumerate(sample_docs, 1):
        print(f"{j}. {doc}")


===== TF-IDF Cluster 0 Sample Documents =====
1. I'm so over Comcast! The worst internet provider. I'm taking online classes and multiple times was late with my assignments because of the power interruptions in my area that lead to poor quality internet service. Definitely switching to Verizon. I'd rather pay $10 extra then dealing w/ Comcast and non stopping internet problems.
2. If I could give them a negative star or no stars on this review I would. I have never worked with any industry with as bad of customer service as Comcast. It is not a matter of money because I make well enough above and beyond to afford their services but they are a legitimate ripoff. I think they are the biggest scam of since the mortgage industry's major meltdown and I hope I move somewhere where Comcast does not exist. The disregard to want to help or do the right thing is honestly astounding. If you have to call, which you do FOR ALL ISSUES - billing, connection/service, adding or removing service, error

In [47]:
for i in range(k):
    print(f"\n===== Word2Vec Cluster {i} Sample Documents =====")
    sample_docs = df_result[df_result['Cluster_Word2Vec'] == i][text_col].head(5).tolist()
    for j, doc in enumerate(sample_docs, 1):
        print(f"{j}. {doc}")


===== Word2Vec Cluster 0 Sample Documents =====
1. I'm so over Comcast! The worst internet provider. I'm taking online classes and multiple times was late with my assignments because of the power interruptions in my area that lead to poor quality internet service. Definitely switching to Verizon. I'd rather pay $10 extra then dealing w/ Comcast and non stopping internet problems.
2. If I could give them a negative star or no stars on this review I would. I have never worked with any industry with as bad of customer service as Comcast. It is not a matter of money because I make well enough above and beyond to afford their services but they are a legitimate ripoff. I think they are the biggest scam of since the mortgage industry's major meltdown and I hope I move somewhere where Comcast does not exist. The disregard to want to help or do the right thing is honestly astounding. If you have to call, which you do FOR ALL ISSUES - billing, connection/service, adding or removing service, err

In [48]:
print("Conclusion:")
print("Text preprocessing was added before TF-IDF and Word2Vec vectorization.")
print("Then KMeans clustering was applied on the complaint texts from customer_complaints_1.csv.")

Conclusion:
Text preprocessing was added before TF-IDF and Word2Vec vectorization.
Then KMeans clustering was applied on the complaint texts from customer_complaints_1.csv.


Text preprocessing was added before both TF-IDF and Word2Vec vectorization. The preprocessing included lowercasing, removing symbols, removing stopwords, and lemmatization. After that, KMeans clustering was applied to the complaint texts from customer_complaints_1.csv. The clustering results show how similar complaint texts can be grouped together based on their content. In general, preprocessing helps improve clustering quality because it removes noise and keeps more meaningful words for vectorization.